# 处理前后并排对比（答辩用）

左边原图、右边处理后的图、第三张是差值热力图。

**为什么不用改硬件**：`MODE_SEL=0` + `PARAM=128` 就是 **×1.0 完全直通**
（增益公式 `v = (c * PARAM) >> 7`，PARAM=128 时 `v = c`）。
所以切一下这两个寄存器就能拿到「原始画面」，抓完再切回自适应模式抓处理后的。

> 注意：这是 **Jupyter 里的并排对比**，不是 HDMI 分屏。
> HDMI 上真分屏需要让视频流在 demosaic 后分叉并加合成逻辑，改动和风险都在硬件侧，
> 详见 `docs/06_split_screen_options.md`。

跑法：先让画面停在你要演示的场景上（别动），然后从上往下跑。


In [ ]:
import os
import sys

import numpy as np
import matplotlib.pyplot as plt

ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT):
    if os.path.isfile(os.path.join(ROOT, "lib", "video_setup.py")):
        break
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)

from lib import video_setup as vs
from lib import video_path as vp
from lib import adaptive
from lib import exposure
from lib.ai_video_regs import AiVideoRegs

# 和采集/演示同一档曝光，否则「原图 vs 处理后」的对比里混进的是曝光差异。
ol, iic, demosaic, vdma0, vdma1, mode = vs.bring_up(
    ROOT, manual_exposure=dict(exposure.WORKPOINT))
vs.stop_all_channels(vdma0, vdma1)
regs = AiVideoRegs(ol.axis_video_proc_0)
print("VERSION = 0x%08X (阶段 %d)" % (regs.version(), regs.version_stage()))

vp.start_capture(vdma0, mode)
regs.wait_new_frame()
print("采集已启动。把画面停在你想要的场景上，然后跑下一格。")


## 抓两帧：原图（直通） + 处理后的图

两次抓帧之间**画面必须不动**，否则差值里混进的是「场景变了」而不是「处理造成的」。


In [ ]:
def grab(mode_sel, param, warm=5):
    regs.set_mode(mode_sel)
    regs.set_param(param)
    for _ in range(warm):
        regs.wait_new_frame()
    return vp.frame_to_rgb(vp.capture_frame(vdma0)).astype(np.float32)


# ---- 1) 原始画面：mode0 + PARAM=128 = ×1.0 直通 ----
raw = grab(0, 128)

# ---- 2) 处理后的画面：按当前特征走一次自适应决策 ----
f = regs.mlp_features()
mlp = regs.mlp_output()
m, p, why = adaptive.decide(f, mlp)
proc = grab(m, p)

print("直通帧 均值=%.1f  std=%.1f" % (raw.mean(), raw.std()))
print("处理帧 均值=%.1f  std=%.1f   (mode=%d PARAM=%d  %s)"
      % (proc.mean(), proc.std(), m, p, why))


## 并排显示 + 差值热力图


In [ ]:
diff = np.abs(proc - raw)
dmag = diff.mean(axis=2)

fig, ax = plt.subplots(1, 3, figsize=(21, 5))
ax[0].imshow(raw.astype(np.uint8))
ax[0].set_title("LEFT  : camera raw (mode0 + PARAM=128 = passthrough)")
ax[1].imshow(proc.astype(np.uint8))
ax[1].set_title("RIGHT : processed  (mode=%d, PARAM=%d)" % (m, p))
im = ax[2].imshow(dmag, cmap="hot")
ax[2].set_title("diff |processed - raw|  mean=%.1f" % dmag.mean())
fig.colorbar(im, ax=ax[2], fraction=0.046)
for a in ax:
    a.axis("off")
plt.tight_layout()
plt.show()


## 量化对比（贴进报告用）

三个指标：亮度（看增益）、对比度 std（看逆光补偿）、水平差分（看锐化）。


In [ ]:
def metrics(a):
    return (float(a.mean()), float(a.std()),
            float(np.abs(np.diff(a, axis=1)).mean()),
            100.0 * float((a >= 255).mean()))


print("%-10s %8s %8s %10s %8s" % ("", "mean", "std", "h-diff", "sat%"))
print("%-10s %8.1f %8.1f %10.2f %8.1f" % (("raw",) + metrics(raw)))
print("%-10s %8.1f %8.1f %10.2f %8.1f" % (("processed",) + metrics(proc)))
print()
print("说明：mode0 看 mean 变化（增益），mode1 看 std 变化（压对比），")
print("      mode2 看 h-diff 变化（锐化）；sat% 是过曝比例，越低越好。")


## 收尾


In [ ]:
regs.set_mode(0)
regs.set_param(192)
vs.stop_all_channels(vdma0, vdma1)
print("[ok] 模式恢复到 0 / PARAM=192")
try:
    ol.free()
    print("[ok] overlay 已释放")
except Exception as e:
    print("[warn] %s" % e)
